[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/fresnel/fresnel.ipynb)

# Fresnel wave surfaces

In a crystal, light of one frequency travelling in one direction can have two wavelengths, one for each polarization: the two sheets of the Fresnel wave surface. This notebook finds them with *extensors*: geometric expressions with an argument left open, which numga then treats as maps it can apply, compose and take apart. A material is a map from the field to its excitation; a wave is a map from a potential to what is left of Maxwell's equations; and a wave can travel where that map loses enough directions, which one adjugate turns into a single polynomial.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.electromagnetism.fresnel import render
from examples.electromagnetism.fresnel.core import sphere, circle

ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antibivector = ga.gatype.antibivector()
Antivector = ga.gatype.antivector()
Constitutive = ga.gatype((Antibivector, Bivector))            # excitation <- field
WaveMap = ga.gatype((Vector, Vector))                       # residual <- potential

observer = mv.t
axes = stack([mv.x, mv.y, mv.z])                            # [3] Vector
# One electric weight per principal axis, and one entry per material.
names = ("Vacuum", "Uniaxial crystal", "Biaxial crystal")
permittivity_x = np.array([1.0, 2.25, 1.44])
permittivity_y = np.array([1.0, 2.25, 2.25])
permittivity_z = np.array([1.0, 3.24, 3.24])
permittivity = np.stack([permittivity_x, permittivity_y, permittivity_z], axis=-1)
reluctivity = np.array([1.0, 1.0, 1.0])
frequency = 1.0
latitudes = 33
longitudes = 2 * (latitudes - 1) + 1
section_samples = 4 * (latitudes - 1) + 1
radius_squared = np.arange(3)
root_signs = np.array([-1.0, 1.0])

## 1. The material as a map

The field of a light wave is a bivector of spacetime, and a material answers it with an excitation. Writing `Bivector` where the field goes leaves it open, so the expression below is the whole response at once, a map `Antibivector <- Bivector`. It weights each of the observer's three electric planes, `axes ^ observer`, by its own permittivity: `planes * (planes | Bivector)` reads the field's part along each plane and returns that plane. The observer's sandwich `observer >> Bivector` keeps the magnetic planes and turns the electric ones around, so half its sum with the identity keeps the magnetic planes alone, and they share one weight.

In [ ]:
# --- math -----------------------------------------------------------------------------
electric_planes = axes ^ observer                                           # [3] Bivector
# Each plane reads the field component along itself; sum the weighted responses over the axes.
electric = (permittivity * electric_planes * (electric_planes | Bivector)).sum(axis=-1)
# The sandwich by the observer keeps the magnetic planes and turns the electric ones around:
# half the sum with the identity keeps the magnetic planes alone.
magnetic = (Bivector + (observer >> Bivector)) / 2
media = (electric + reluctivity * magnetic).dual()                          # [materials] Constitutive

## 2. When the potential map has a wave

A potential `A` gives the field `k ^ A`, and Maxwell's other equation asks that `k ^ material(k ^ A)` vanish. With the potential left open this is a map `Vector <- Vector`, the potential map. It always loses one direction, since a potential along `k` itself gives no field; a wave is a second potential it loses. A map loses two directions exactly when it sends every volume, three potentials wedged together, to nothing, and `volumes = wave.outermorphism(Antivector)` is that action on volumes. Its adjugate, the map on vectors that keeps every incidence across it, `volumes.adjugate()(c) & v == c & volumes(v)`, then holds the two factors of `k` and one number more: the Fresnel quartic, zero where a wave can travel.

In tensor-index notation the factorization reads as $\operatorname{adj}(M)_{ab}=\lambda(k)k_a k_b$, up to the dualization convention, with dispersion equation $\lambda(k)=0$; see [Itin](https://arxiv.org/abs/0908.0922).

In [ ]:
def fresnel(medium: Constitutive, wavevector: Vector) -> Scalar:
    # The open Vector is the potential. Its wedge with wavevector is the field,
    # and wedging the excitation with wavevector imposes the other Maxwell equation.
    field = wavevector ^ Vector                                              # [...] Bivector <- Vector
    wave = (wavevector ^ medium(field)).dual()                                # [...] WaveMap

    # wave(wavevector) vanishes for every candidate. A physical wave requires
    # another null direction, so all three-dimensional volume images vanish too.
    adjugate = wave.outermorphism(Antivector).adjugate()                       # [...] WaveMap
    measured_frequency = observer | wavevector
    return -(observer | adjugate(observer)) / measured_frequency.squared()

## 3. The two sheets

Hold the frequency fixed and look along one spatial direction at a time. For these crystals at rest the quartic is even in the length of the spatial wavevector, so as a function of its square it is a quadratic: three samples fix it, and its two positive roots are the two sheets.

In [ ]:
def radial_sheets(medium: Constitutive, directions: Vector) -> Vector:
    probes = observer * frequency + directions[..., None] * np.sqrt(radius_squared)
    values = fresnel(medium[..., None], probes)
    constant, unit, twice = (values[..., i] for i in range(3))
    quadratic = (twice - 2 * unit + constant) / 2
    linear = unit - constant - quadratic

    # Where the sheets coincide the discriminant vanishes, up to round-off of either sign.
    discriminant = (linear.squared() - 4 * quadratic * constant).abs().square_root()
    radii_squared = (-linear[..., None] + discriminant[..., None] * root_signs) / (2 * quadratic[..., None])
    return directions[..., None] * radii_squared.square_root()


# The samplers turn a spatial direction with rotors, over a sphere and an xz meridian.
directions = sphere(latitudes, longitudes)                                   # [latitudes, longitudes] Vector
section_directions = circle(section_samples)                                 # [section_samples] Vector
sheets = radial_sheets(media[:, None, None], directions)                      # [materials, latitudes, longitudes, sheets] Vector
sections = radial_sheets(media[:, None], section_directions)                  # [materials, section_samples, sheets] Vector
render.comparison(sheets, sections, names);

In [ ]:
# checks
# Every wavevector on the sheets solves the quartic, and its potential map loses the potential along it.
wavevectors = observer * frequency + sheets                                     # [materials, latitudes, longitudes, sheets] Vector
np.testing.assert_allclose(fresnel(media[:, None, None, None], wavevectors).kernel, 0.0, atol=1e-9)
wave = (wavevectors ^ media[:, None, None, None](wavevectors ^ Vector)).dual()   # [materials, latitudes, longitudes, sheets] Vector <- Vector
np.testing.assert_allclose(wave(wavevectors).kernel, 0.0, atol=1e-11)


The vacuum sheets coincide. A uniaxial crystal makes them touch along one optic axis; a biaxial crystal has two. The lower panels cut the surfaces through the xz plane, at frequency one in units of the vacuum light speed. Run `python -m examples.electromagnetism.fresnel.scenarios` to save the figure and animation in `plots/`.